# Chapter 04 Exercises

This chapter's ideas are the ones that make later bugs explainable. **Predict
before you run** — especially in Section B, where several answers are
counter-intuitive.

- **A** — concept checks
- **B** — predict the output (the important one)
- **C** — find and fix the bug
- **D** — trace the references
- **E** — write it yourself
- **F** — self-assessment

## Section A — Concept checks

In [ ]:
questions = [
    (
        "A1. Why is 'a variable is a box holding a value' wrong in Python?",
        "A name is a LABEL pointing at an object, not a container. "
        "Assignment repoints the label; it never puts a value into a box. "
        "This is why a = b gives two names for one object.",
    ),
    (
        "A2. What are the three properties of every object?",
        "Identity (id, never changes), type (never changes), and value "
        "(changes only if the object is mutable).",
    ),
    (
        "A3. Is Python pass by value or pass by reference?",
        "Neither. It is pass by ASSIGNMENT - the parameter becomes another "
        "name for the caller's object. Mutating it is visible to the caller; "
        "rebinding it is not.",
    ),
    (
        "A4. Why can a tuple be a dict key but a list cannot?",
        "Dict keys must be hashable, and hashability requires the value "
        "never changes. A list is mutable, so its hash could change and the "
        "dict could never find the entry again.",
    ),
    (
        "A5. Why can a tuple containing a list NOT be a dict key?",
        "Immutability is shallow. The tuple's slots are fixed, but the list "
        "inside can still change, so the tuple is unhashable.",
    ),
    (
        "A6. When do x += y and x = x + y behave differently?",
        "For mutable types. += calls __iadd__ and mutates in place, so "
        "aliases see the change. = + calls __add__ and builds a new object. "
        "For immutable types there is no __iadd__, so both rebind.",
    ),
    (
        "A7. What can reference counting NOT free, and what handles it?",
        "Reference cycles - each object keeps the other's count above zero. "
        "The generational garbage collector finds unreachable cycles, but "
        "runs periodically, so that cleanup is not deterministic.",
    ),
    (
        "A8. Why should you never rely on __del__ for cleanup?",
        "Its timing is not guaranteed: cycles, interpreter shutdown, live "
        "tracebacks and other implementations all delay or skip it. Use a "
        "context manager (with) instead.",
    ),
    (
        "A9. Does Python have constants?",
        "No. UPPER_SNAKE_CASE is a convention only. typing.Final is checked "
        "by type checkers, not at runtime. Enum gives real runtime "
        "protection, as do immutable types like tuple and frozenset.",
    ),
    (
        "A10. Why does a, b = b, a work without a temporary variable?",
        "The right-hand side is evaluated completely into a tuple before any "
        "assignment happens. Both old values are on the stack before either "
        "name is stored.",
    ),
]

for question, answer in questions:
    print(question)
    print("   ", answer)
    print("")

## Section B — Predict the output

Write your answer down first. Several of these catch experienced programmers.

In [ ]:
print("B1. What does this print?")
print("      a = [1, 2]")
print("      b = a")
print("      b.append(3)")
print("      print(a)")
print("")

a = [1, 2]
b = a
b.append(3)
print("   RESULT:", a)
print("   WHY: b is not a copy. One object, two names.")

In [ ]:
print("B2. What does this print?")
print("      a = [1, 2]")
print("      b = a")
print("      b = b + [3]")
print("      print(a)")
print("")

a = [1, 2]
b = a
b = b + [3]
print("   RESULT:", a)
print("   WHY: `b + [3]` built a NEW list and rebound b. `a` still points")
print("        at the original object.")

In [ ]:
print("B3. Now with += instead. What changes?")
print("      a = [1, 2]")
print("      b = a")
print("      b += [3]")
print("      print(a)")
print("")

a = [1, 2]
b = a
b += [3]
print("   RESULT:", a)
print("   WHY: += calls __iadd__ which mutates IN PLACE. Same syntax as")
print("        B2 in spirit, opposite result.")

In [ ]:
print("B4. What does each call return?")
print("      def add(item, target=[]):")
print("          target.append(item)")
print("          return target")
print("")

def add(item, target=[]):
    target.append(item)
    return target

print("   call 1:", add("a"))
print("   call 2:", add("b"))
print("   call 3:", add("c"))
print("")
print("   WHY: the default list was created ONCE when def ran.")
print("        Every call shares it. Use None as the sentinel instead.")

In [ ]:
print("B5. What does this print?")
print("      grid = [[0] * 3] * 3")
print("      grid[0][0] = 1")
print("      print(grid)")
print("")

grid = [[0] * 3] * 3
grid[0][0] = 1
print("   RESULT:", grid)
print("   WHY: `* 3` repeated the REFERENCE three times, not the row.")
print("        All three entries are the same list object.")
print("")

correct = [[0] * 3 for _ in range(3)]
correct[0][0] = 1
print("   CORRECT:", correct)

In [ ]:
print("B6. True or False for each?")
print("")

# Built at runtime so the compiler cannot fold them into one constant.
small_one = int("100")
small_two = int("100")
large_one = int("1000")
large_two = int("1000")

print("   int('100')  is int('100') :", small_one is small_two)
print("   int('1000') is int('1000'):", large_one is large_two)
print("   int('1000') == int('1000'):", large_one == large_two)
print("")
print("   WHY: CPython caches integers from -5 to 256. 100 is cached, so")
print("        both names share one object. 1000 is not. Never use `is`")
print("        to compare values.")

In [ ]:
print("B7. What does this print?")
print("      t = (1, [2, 3])")
print("      t[1].append(4)")
print("      print(t)")
print("")

t = (1, [2, 3])
t[1].append(4)
print("   RESULT:", t)
print("   WHY: the tuple's slots are fixed, but the list inside is still")
print("        mutable. Immutability is SHALLOW.")
print("")

try:
    hash(t)
except TypeError as error:
    print("   And it cannot be a dict key:", error)

In [ ]:
print("B8. What does the class attribute do?")
print("")

class Team:
    members = []          # shared by every instance

    def add(self, name):
        self.members.append(name)

team_a = Team()
team_b = Team()
team_a.add("Asha")

print("   team_a.members:", team_a.members)
print("   team_b.members:", team_b.members)
print("   same object?   ", team_a.members is team_b.members)
print("")
print("   WHY: the list belongs to the CLASS, not to each instance.")
print("        Create it in __init__ to give each instance its own.")

## Section C — Find and fix the bug

Each cell has working-looking code with a real defect.

In [ ]:
# C1 - a function that damages its caller's data.

def apply_discount_broken(prices, percent):
    """Apply a discount - but destroys the input."""
    for index in range(len(prices)):
        prices[index] = prices[index] * (1 - percent / 100)
    return prices


original = [100.0, 200.0]
discounted = apply_discount_broken(original, 10)

print("C1. BROKEN")
print("   returned:", discounted)
print("   original:", original, "<- also changed")
print("   same object?", discounted is original)


def apply_discount_fixed(prices, percent):
    """Return a NEW list, leaving the input untouched."""
    # A comprehension builds a new list rather than mutating.
    return [price * (1 - percent / 100) for price in prices]


original = [100.0, 200.0]
discounted = apply_discount_fixed(original, 10)

print("")
print("   FIXED")
print("   returned:", discounted)
print("   original:", original, "<- intact")

In [ ]:
# C2 - removing items while iterating.

numbers = [1, 2, 3, 4, 5, 6]

buggy = list(numbers)
for value in buggy:
    if value % 2 == 0:
        buggy.remove(value)

print("C2. BROKEN")
print("   input: ", numbers)
print("   result:", buggy, "<- some evens survived")
print("   WHY: removing shifts the remaining items, so the loop skips one.")

# Fix: build a new list instead of mutating the one you are reading.
fixed = [value for value in numbers if value % 2 != 0]
print("")
print("   FIXED:", fixed)

# Alternative fix: iterate over a copy.
also_fixed = list(numbers)
for value in list(also_fixed):
    if value % 2 == 0:
        also_fixed.remove(value)
print("   ALSO FIXED (iterate a copy):", also_fixed)

In [ ]:
# C3 - a class that shares state with its caller.

class Basket:
    """Stores the list it is handed - an alias, not a copy."""

    def __init__(self, items):
        self.items = items


caller_items = ["apple"]
basket = Basket(caller_items)
caller_items.append("injected from outside")

print("C3. BROKEN")
print("   basket.items:", basket.items, "<- the caller changed it")


class SafeBasket:
    """Copies the incoming list."""

    def __init__(self, items):
        # list() creates an independent copy.
        self.items = list(items)


caller_items = ["apple"]
safe = SafeBasket(caller_items)
caller_items.append("injected from outside")

print("")
print("   FIXED")
print("   basket.items:", safe.items, "<- unaffected")

## Section D — Trace the references

Work out the reference count at each step before running.

In [ ]:
import sys

def refcount(obj):
    """Real reference count, excluding getrefcount's own temporary."""
    return sys.getrefcount(obj) - 1


print("Predict the count after each line:")
print("")

data = ["x"]
print("   data = ['x']                 ->", refcount(data))

alias = data
print("   alias = data                 ->", refcount(data))

container = [data, data]
print("   container = [data, data]     ->", refcount(data))

lookup = {"a": data, "b": data}
print("   lookup = {'a': data, ...}    ->", refcount(data))

del alias
print("   del alias                    ->", refcount(data))

container.pop()
print("   container.pop()              ->", refcount(data))

del lookup
print("   del lookup                   ->", refcount(data))

print("")
print("Each name, list slot and dict value holds one reference.")

## Section E — Write it yourself

Create these as real `.py` files and run them.

In [ ]:
tasks = [
    ("E1", "identity.py",
     "Write a function report(name, obj) printing id, type and value. "
     "Call it on a list before and after append, and before and after "
     "rebinding. Explain which operations preserved identity."),
    ("E2", "mutation.py",
     "Write two functions taking a list: one that mutates it, one that "
     "rebinds. Prove with id() and output which the caller sees."),
    ("E3", "defaults.py",
     "Write the mutable-default bug, demonstrate it over three calls, "
     "then fix it with None. Print __defaults__ for both versions."),
    ("E4", "copying.py",
     "Take a nested list. Make an alias, a shallow copy and a deep copy. "
     "Mutate an inner element and show which of the three changed."),
    ("E5", "cycles.py",
     "Build a two-object reference cycle with __del__ methods. Use "
     "gc.disable() to show nothing is freed, then gc.collect()."),
    ("E6", "constants.py",
     "Define constants as UPPER_CASE, as Final, and as an Enum. Attempt "
     "to modify each and report what Python allows."),
    ("E7", "unpacking.py",
     "Unpack a list of five numbers six different ways, including nested "
     "and starred forms. Include a three-way swap."),
]

print("Write these as real .py files:")
print("")
for number, filename, description in tasks:
    print(f"  {number}. {filename}")
    print(f"      {description}")
    print("")

print("Rules:")
print("   - a comment above every meaningful line")
print("   - run `ruff check` on each and fix everything reported")
print("   - use id() and `is` to prove your claims, not assertions")

## Section F — Self-assessment

In [ ]:
checklist = [
    "I can explain why a variable is a label, not a box.",
    "I know the three properties of an object and which can change.",
    "I can explain pass by assignment, and predict what a caller sees.",
    "I know which built-in types are mutable and which are not.",
    "I can explain why immutability is shallow.",
    "I know why only immutable objects are hashable.",
    "I can explain when += differs from = +.",
    "I can spot the mutable default argument bug.",
    "I know why [[0] * 3] * 3 is wrong.",
    "I can explain reference counting and what it cannot free.",
    "I know why __del__ is unreliable and what to use instead.",
    "I know Python has no real constants, and the three alternatives.",
    "I can unpack nested structures and use * to collect a remainder.",
    "I can explain why a, b = b, a needs no temporary variable.",
]

print("Ready for Chapter 05?")
print("-" * 70)
for item in checklist:
    print("   [ ]", item)

print("")
print("All ticked? Chapter 05 - Data Types: Numbers and Booleans.")
print("Starting with why 0.1 + 0.2 is not 0.3.")